# Drug-only outlier testing against the legacy model

This notebook isolates the drug-type-and-quantity dimension of sentencing. For each verified charge it compares the court's drug-based starting sentence against the legacy model's `starting_point` (which is computed purely from drug types and quantities via `get_starting_point`).

No aggravating or mitigating factors are considered on either side of the comparison.

Court value selection (per user request):
- Prefer `trial.starting_point.total_months`.
- Fall back to `trial.sentence_after_role.total_months` when `starting_point` is not available.
- Skip the trial when neither is present.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv

repo_root = Path.cwd().resolve()
if not (repo_root / 'featureExtraction').exists():
    repo_root = repo_root.parent

for env_path in (
    repo_root / 'featureExtraction' / '.env',
    repo_root / 'featureVerification' / '.env.local',
    repo_root / '.env',
):
    if env_path.exists():
        load_dotenv(env_path)

from evaluate_verified_sentences import build_model_input, get_collection, get_total_months
from legacy_model import DkPredictor

# Drug-amount columns from the 17-element legacy model input vector (indices 7-14).
# The legacy model's starting_point depends ONLY on these values, so this notebook
# compares it against the court's starting_point / sentence_after_role (also drug-only).
DRUG_INPUT_COLUMNS = [
    'cocaine_amount',
    'heroin_amount',
    'meth_amount',
    'ketamine_amount',
    'nimetazepam_amount',
    'ecstasy_amount',
    'cannabisresin_amount',
    'herbalcannabis_amount',
]

OUTLIER_THRESHOLD_MONTHS = 6

verified_collection, _ = get_collection()
predictor = DkPredictor()
query = {'is_verified': True}
projection = {
    'filename': 1,
    'judgement.neutral_citation': 1,
    'exclude': 1,
    'remarks': 1,
    'trials': 1,
}
docs = list(verified_collection.find(query, projection))


def format_drugs(trial: dict[str, Any]) -> str:
    parts = []
    for drug in trial.get('drugs') or []:
        drug_type = drug.get('drug_type')
        quantity = drug.get('quantity')
        if drug_type:
            parts.append(f"{drug_type}:{quantity}")
    return '; '.join(parts)


def get_court_starting_months(
    trial: dict[str, Any],
) -> tuple[int | None, str | None]:
    """Return (total_months, source_label) for the court's drug-based sentence.

    Prefer starting_point; fall back to sentence_after_role when starting_point
    is missing. Return (None, None) when neither is available so the caller can
    skip the trial.
    """
    starting_point = trial.get('starting_point')
    if starting_point:
        return get_total_months(starting_point), 'starting_point'
    sentence_after_role = trial.get('sentence_after_role')
    if sentence_after_role:
        return get_total_months(sentence_after_role), 'sentence_after_role'
    return None, None


rows: list[dict[str, Any]] = []
skipped_no_court_value = 0
for doc in docs:
    trials = (doc.get('trials') or {}).get('trials') or []
    for index, trial in enumerate(trials):
        court_starting_months, source_label = get_court_starting_months(trial)
        if court_starting_months is None:
            skipped_no_court_value += 1
            continue

        model_input = build_model_input(trial)
        explanation = predictor.explain(model_input)
        legacy_starting_months = int(explanation['starting_point'])
        difference_months = legacy_starting_months - court_starting_months

        if abs(difference_months) < OUTLIER_THRESHOLD_MONTHS:
            continue

        row: dict[str, Any] = {
            'neutral_citation': (doc.get('judgement') or {}).get('neutral_citation'),
            'exclude_case': bool(doc.get('exclude')),
            'trial_index': index,
            'charge_no': (trial.get('charge_type') or {}).get('charge_no'),
            'charge_name': (trial.get('charge_type') or {}).get('charge_name'),
            'drugs': format_drugs(trial),
        }
        for name, value in zip(DRUG_INPUT_COLUMNS, model_input[7:15]):
            row[name] = value
        row.update({
            'court_starting_months': court_starting_months,
            'court_value_source': source_label,
            'legacy_model_starting_months': legacy_starting_months,
            'difference_months': difference_months,
            'absolute_difference_months': abs(difference_months),
            'remarks': doc.get('remarks'),
        })
        rows.append(row)

outlier_df = pd.DataFrame(rows).sort_values(
    ['difference_months', 'court_starting_months'],
    ascending=[False, False],
).reset_index(drop=True)

output_dir = repo_root / 'notebooks'
output_dir.mkdir(exist_ok=True)
try:
    outlier_df.to_excel(output_dir / 'drug_only_outlier_review.xlsx', index=False)
except Exception as exc:
    print(f'Excel export skipped: {exc}')

print(f'Skipped {skipped_no_court_value} trials with no starting_point or sentence_after_role')
print(f'Found {len(outlier_df)} drug-only outlier candidates using a threshold of {OUTLIER_THRESHOLD_MONTHS} months')
outlier_df.head()

Skipped 1 trials with no starting_point or sentence_after_role
Found 884 drug-only outlier candidates using a threshold of 6 months


,neutral_citation,exclude_case,trial_index,charge_no,charge_name,drugs,cocaine_amount,heroin_amount,meth_amount,ketamine_amount,nimetazepam_amount,ecstasy_amount,cannabisresin_amount,herbalcannabis_amount,court_starting_months,court_value_source,legacy_model_starting_months,difference_months,absolute_difference_months,remarks
0,[2022] HKCFI 183,True,0,1,Trafficking in a dangerous drug,Cocaine:14300,14300.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,starting_point,356,356,356,Ms Ramirez found to have no case to answer for...
1,[2023] HKCFI 2256,False,1,2,Conspiracy to traffic in dangerous drugs,Methamphetamine:10860,0.00,0.0,10860.0,0.0,0.0,0.0,0.0,0.0,0,starting_point,341,341,341,
2,[2021] HKCFI 3788,True,0,1,Trafficking in a dangerous drug,Cocaine:6000,6000.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,starting_point,320,320,320,not convicted for trafficking in dangerous drgus
3,[2022] HKCFI 3370,True,1,1,Trafficking in a dangerous drug,Methamphetamine:3158.5,0.00,0.0,3158.5,0.0,0.0,0.0,0.0,0.0,0,starting_point,301,301,301,D2 has not been convicted of drug trafficking.
4,[2024] HKCFI 3175,True,1,2,Trafficking in a dangerous drug,Cocaine:968.61,968.61,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,starting_point,262,262,262,There is only the final total imprisonment sen...
